<a href="https://colab.research.google.com/github/PauloRadatz/opendss-python-examples/blob/main/presentations/engr330_charleston_southern_university/thermal_gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thermal Hosting Capacity Example

This notebook illustrates a hosting capacity workflow based on **thermal limits** (line and transformer loading) using OpenDSS and Python.

In [1]:
!pip install py-dss-interface
!pip install py-dss-toolkit
!git clone https://github.com/PauloRadatz/opendss-python-examples.git
%cd opendss-python-examples

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 3.7 MB/s eta 0:00:00
Cloning into 'opendss-python-examples'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 63 (delta 13), reused 50 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 246.89 KiB | 6.50 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/opendss-python-examples


In [2]:
import pathlib
import numpy as np
import py_dss_interface
from py_dss_toolkit import dss_tools

# --- Helper functions ---
STEP_KW = 100
MAX_KW = 10000
THERMAL_LIMIT_PER = 100

def add_gen(dss, gen_bus, gen_kv):
    for gen in gen_bus:
        dss.text(f"new generator.{gen} phases=3 bus1={gen_bus[gen]} kv={gen_kv[gen]} "
                 f"kw=0.0001 pf=1 Vminpu=0.7 Vmaxpu=1.2")

def increase_gen(dss, gen_kw):
    for gen, kw in gen_kw.items():
        dss.text(f"Edit generator.{gen} kw={kw}")

def solve_powerflow(dss):
    dss.text("solve")

def check_thermal_violation(dss):
    for element in dss.circuit.elements_names:
        if element.split(".")[0].lower() in ["line", "transformer"]:
            dss.circuit.set_active_element(element)
            current = dss.cktelement.currents_mag_ang
            rating_current = dss.cktelement.norm_amps
            num_terminal = 1  # we could look at both terminals
            phase_currents = current[:int(len(current) * num_terminal / 2):2]
            if phase_currents and rating_current > 0:
                if max(phase_currents) / rating_current > THERMAL_LIMIT_PER / 100.0:
                    return True
    return False

def set_load_level_condition(dss, load_mult):
    dss.text(f"set loadmult={load_mult}")

def result_centralized_info(bus, metric, hc_value, load_condition, existing_gen, device_type):
    return f"""Summary of the Centralized Hosting Capacity Analysis

Feeder Conditions:
    Load level condition = {load_condition}
    Existing Gen considered = {existing_gen}

Hosting Capacity:
    Bus = {bus}
    Value = {hc_value} kW
    Device Type = {device_type}
    Metric = {metric}"""

## Define feeder and create OpenDSS object and connect to dss_tools

In [3]:
start_dir = pathlib.Path.cwd().resolve()
repo_root = next(
    parent for parent in [start_dir, *start_dir.parents]
    if (parent / "feeder_models").exists()
)
dss_file = repo_root / "feeder_models" / "8bus" / "Master.dss"
dss = py_dss_interface.DSS()

dss_tools.update_dss(dss)

print(f"Using feeder file: {dss_file}")

Using feeder file: /content/opendss-python-examples/feeder_models/8bus/Master.dss


## Compile feeder and set operating condition

In [4]:
dss.text(f'compile "{dss_file}"')

load_mult = 0.2
set_load_level_condition(dss, load_mult=load_mult)

print(f"Load multiplier set to {load_mult}.")

Load multiplier set to 0.2.


## Select the bus where generation will be connected

In [5]:
bus = "4"

dss.circuit.set_active_bus(bus)
kv = dss.bus.kv_base * np.sqrt(3)

gen_bus = {"gen": dss.bus.name}
gen_kv = {"gen": kv}

print(f"Selected bus: {dss.bus.name}")
print(f"Generator base voltage: {kv:.4f} kV")

Selected bus: 4
Generator base voltage: 12.4700 kV


## Add the generator

In [6]:
add_gen(dss, gen_bus, gen_kv)

print("Generator added successfully.")

Generator added successfully.


## Increase generation until thermal violation is detected

In [7]:
hosting_capacity_value = None

i = 0
while i * STEP_KW < MAX_KW:
    i += 1
    i_kw = i * STEP_KW
    gen_kw = {"gen": i_kw}

    increase_gen(dss, gen_kw)
    solve_powerflow(dss)

    if check_thermal_violation(dss):
        hosting_capacity_value = (i - 1) * STEP_KW
        print(f"Thermal violation detected at {i_kw} kW.")
        break

if hosting_capacity_value is None:
    hosting_capacity_value = MAX_KW
    print("No thermal violation detected within the tested range.")

Thermal violation detected at 500 kW.


## Display the result

In [8]:
result = result_centralized_info(
    bus,
    "Thermal",
    hosting_capacity_value,
    "offpeak",
    False,
    "Generator"
)

print(result)

Summary of the Centralized Hosting Capacity Analysis

Feeder Conditions:
    Load level condition = offpeak
    Existing Gen considered = False

Hosting Capacity:
    Bus = 4
    Value = 400 kW
    Device Type = Generator
    Metric = Thermal


## Checking the feeder with the generator sized to the hosting capacity value

In [9]:
i_kw = hosting_capacity_value
gen_kw = {"gen": i_kw}

increase_gen(dss, gen_kw)
solve_powerflow(dss)

In [10]:
dss_tools.interactive_view.voltage_profile()

In [11]:
sub_marker = dss_tools.interactive_view.circuit_get_bus_marker("0", "square", 12, "black")
gen_marker = dss_tools.interactive_view.circuit_get_bus_marker("4", "circle", 10, "red")
dss_tools.interactive_view.circuit_plot(parameter="thermal violations", bus_markers=[sub_marker, gen_marker])

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [pauloradatz.me](https://www.pauloradatz.me)
- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)